# Data and Partitions

Loads MNIST and generates partition index files for IID, pathological shards, Dirichlet, quantity skew, and feature skew.

In [1]:
import os
import numpy as np
import torch
from src.utils import seed_everything, load_mnist
from src.partition import (
    iid_partition,
    pathological_partition,
    dirichlet_partition,
    quantity_skew_partition,
    feature_skew_partition,
    get_label_distribution,
    save_partition
)

seed_everything(42)
os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)
(train_x, train_y), (test_x, test_y) = load_mnist('data')
print('MNIST train samples:', len(train_y), 'test samples:', len(test_y))

MNIST train samples: 60000 test samples: 10000


Create IID partition with 100 clients, 600 samples each.

In [2]:
iid_parts = iid_partition(train_y, num_clients=100, seed=42)
save_partition(iid_parts, 'data/iid_k100.npz')
iid_dist = get_label_distribution(train_y, iid_parts)
print('IID client 0 size:', len(iid_parts[0]), 'label counts:', iid_dist[0])

IID client 0 size: 600 label counts: [58 63 60 70 66 50 58 61 66 48]


Create pathological non-IID partition using 200 shards, assigning 2 shards per client.

In [3]:
patho_parts = pathological_partition(train_y, num_clients=100, num_shards=200, seed=42)
save_partition(patho_parts, 'data/pathological_k100.npz')
patho_dist = get_label_distribution(train_y, patho_parts)
print('Pathological client 0 size:', len(patho_parts[0]), 'unique classes:', np.count_nonzero(patho_dist[0]))

Pathological client 0 size: 600 unique classes: 2


Create Dirichlet label skew partitions for alpha in 0.1, 0.5, and 10.0.

In [4]:
for alpha in [0.1, 0.5, 10.0]:
    dir_parts = dirichlet_partition(train_y, num_clients=100, alpha=alpha, seed=42)
    alpha_str = str(alpha).replace('.', '')
    save_partition(dir_parts, f'data/dirichlet_a{alpha_str}.npz')
    d_dist = get_label_distribution(train_y, dir_parts)
    print(f'Dirichlet alpha={alpha} client 0 unique classes:', np.count_nonzero(d_dist[0]))

Dirichlet alpha=0.1 client 0 unique classes: 6
Dirichlet alpha=0.5 client 0 unique classes: 9
Dirichlet alpha=10.0 client 0 unique classes: 10


Create quantity skew and feature skew partitions.

In [5]:
qty_parts = quantity_skew_partition(train_y, num_clients=100, beta=0.5, seed=42)
save_partition(qty_parts, 'data/quantity_skew.npz')
sizes = [len(v) for v in qty_parts.values()]
print('Quantity skew client sizes: min =', min(sizes), 'max =', max(sizes))

feat_parts, feat_x = feature_skew_partition(train_x, train_y, num_clients=100, seed=42)
save_partition(feat_parts, 'data/feature_skew.npz')
print('Feature skew generated across', len(feat_parts), 'clients')

Quantity skew client sizes: min = 0 max = 4172


Feature skew generated across 100 clients
